In [10]:
import os
import time
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter

# 1. Setup Rate Limiter (Corrected Argument)
# 0.1 requests per second = 1 request every 10 seconds (6 RPM)
# This is very safe for the Gemini Free Tier
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1, 
    check_every_n_seconds=0.1, 
    max_bucket_size=1
)

# 2. Initialize using your authorized ID
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash", 
    temperature=0,
    rate_limiter=rate_limiter,
    max_retries=3
)

# 3. Simple test chain
prompt = ChatPromptTemplate.from_template("Summarize this: {context}")
chain = prompt | llm | StrOutputParser()

print("🚀 Connecting to models/gemini-2.5-flash with rate limiting...")
try:
    response = chain.invoke({"context": "Rate limiting helps manage API quotas."})
    print("\n✅ SUCCESS!")
    print("💡 Result:", response)
except Exception as e:
    print(f"\n❌ Error: {e}")

🚀 Connecting to models/gemini-2.5-flash with rate limiting...

✅ SUCCESS!
💡 Result: Rate limiting is used to manage API quotas.


In [12]:
pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.rate_limiters import InMemoryRateLimiter

# 1. Load the key explicitly
load_dotenv()
my_key = os.getenv("GOOGLE_API_KEY")

# 2. Setup the Rate Limiter (Safety first)
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1, 
    check_every_n_seconds=0.1, 
    max_bucket_size=1
)

# 3. Initialize LLM with EXPLICIT api_key
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash", 
    google_api_key=my_key,  # Pass it directly here
    temperature=0,
    rate_limiter=rate_limiter
)

# 4. Create the Compressor
# This uses your limited LLM to filter text sequentially
compressor = LLMChainExtractor.from_llm(llm)

# 5. Preprocessing Example
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
raw_text = "Mars is the fourth planet. Jupiter is the largest. Saturn has rings."
docs = [Document(page_content=x) for x in text_splitter.split_text(raw_text)]

print("🚀 Connection and API Key validated.")
print(f"✂️ Compressing {len(docs)} chunks for query...")

try:
    compressed_docs = compressor.compress_documents(docs, query="Tell me about Mars")
    print("\n✅ Compression Successful!")
    for doc in compressed_docs:
        print(f"Snippet: {doc.page_content}")
except Exception as e:
    print(f"❌ Error: {e}")

🚀 Connection and API Key validated.
✂️ Compressing 1 chunks for query...

✅ Compression Successful!
Snippet: Mars is the fourth planet.
